# Coffee Roasting Classification with TensorFlow

A small binary-classification neural network using roasting **temperature** and **duration** as inputs.

This portfolio version keeps the TensorFlow workflow self-contained and removes the external plotting/helper-file dependency.


In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)
np.set_printoptions(precision=2, suppress=True)


## Create the coffee-roasting dataset

The dataset is generated synthetically from temperature and roasting duration, so no external CSV file is required.


In [ ]:
def make_coffee_data(seed=2):
    rng = np.random.default_rng(seed)
    X = rng.random(400).reshape(-1, 2)
    X[:, 1] = X[:, 1] * 4 + 11.5
    X[:, 0] = X[:, 0] * (285 - 150) + 150

    Y = np.zeros(len(X), dtype=int)

    for i, (temperature, duration) in enumerate(X):
        upper_duration = -3 / (260 - 175) * temperature + 21
        good_roast = (
            175 < temperature < 260
            and 12 < duration < 15
            and duration <= upper_duration
        )
        Y[i] = int(good_roast)

    return X, Y.reshape(-1, 1)

X, Y = make_coffee_data()
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Positive examples:", int(Y.sum()))


In [ ]:
def plot_coffee_data(X, Y):
    y = Y.reshape(-1)
    plt.figure(figsize=(7, 5))
    plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="x", label="Good roast")
    plt.scatter(
        X[y == 0, 0],
        X[y == 0, 1],
        marker="o",
        facecolors="none",
        label="Bad roast",
    )
    plt.xlabel("Temperature (Celsius)")
    plt.ylabel("Duration (minutes)")
    plt.title("Coffee Roasting Dataset")
    plt.legend()
    plt.show()

plot_coffee_data(X, Y)


## Normalize the inputs


In [ ]:
print(
    f"Temperature max/min before normalization: "
    f"{np.max(X[:, 0]):.2f} / {np.min(X[:, 0]):.2f}"
)
print(
    f"Duration max/min before normalization: "
    f"{np.max(X[:, 1]):.2f} / {np.min(X[:, 1]):.2f}"
)

norm_layer = tf.keras.layers.Normalization(axis=-1)
norm_layer.adapt(X)
Xn = norm_layer(X)


## Build the neural network

The network has two inputs, a hidden layer with three sigmoid units, and one sigmoid output unit.


In [ ]:
tf.random.set_seed(1234)

model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation="sigmoid", name="layer1"),
    Dense(1, activation="sigmoid", name="layer2"),
])

model.summary()


In [ ]:
L1_num_params = 2 * 3 + 3
L2_num_params = 3 * 1 + 1

print("Layer 1 parameters:", L1_num_params)
print("Layer 2 parameters:", L2_num_params)
print("Total parameters:", L1_num_params + L2_num_params)


## Train the model


In [ ]:
# Repeat the small synthetic dataset to provide more update steps during training.
Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.BinaryCrossentropy(),
)

history = model.fit(Xt, Yt, epochs=10, verbose=0)

plt.plot(history.history["loss"])
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Training Loss")
plt.show()


## Inspect predictions


In [ ]:
probabilities = model.predict(Xn, verbose=0)
y_hat = (probabilities >= 0.5).astype(int)

accuracy = np.mean(y_hat == Y)
print(f"Training-set classification accuracy: {accuracy:.3f}")
print("First 10 probabilities:")
print(probabilities[:10].reshape(-1))
print("First 10 decisions:")
print(y_hat[:10].reshape(-1))


In [ ]:
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)


## Visualize the learned decision surface


In [ ]:
temperature = np.linspace(150, 285, 120)
duration = np.linspace(11.5, 15.5, 120)
xx, yy = np.meshgrid(temperature, duration)

grid = np.column_stack([xx.ravel(), yy.ravel()])
grid_n = norm_layer(grid)
grid_prob = model.predict(grid_n, verbose=0).reshape(xx.shape)

plt.figure(figsize=(8, 5))
contour = plt.contourf(xx, yy, grid_prob, levels=np.linspace(0, 1, 11), alpha=0.5)
plt.colorbar(contour, label="Predicted probability")

y = Y.reshape(-1)
plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="x", label="Good roast")
plt.scatter(
    X[y == 0, 0],
    X[y == 0, 1],
    marker="o",
    facecolors="none",
    label="Bad roast",
)

plt.xlabel("Temperature (Celsius)")
plt.ylabel("Duration (minutes)")
plt.title("TensorFlow Neural Network Decision Surface")
plt.legend()
plt.show()


## Takeaway

This notebook demonstrates a complete small TensorFlow classification workflow: synthetic data generation, normalization, neural-network construction, training with Adam and binary cross-entropy, threshold-based classification, parameter inspection, and decision-surface visualization.
